In [2]:
import pandas as pd
import numpy as np
import psycopg
import hashlib

from decimal import Decimal
from pathlib import Path



In [9]:

from getpass import getpass
import psycopg

pg_password = getpass("PostgreSQL password: ")

conn = psycopg.connect(
    host="localhost",
    port=5432,
    dbname="sales_etl",
    user="postgres",
    password=pg_password
)

print("PostgreSQL connected successfully!")
bronze_df = pd.read_sql_query(
    """
    SELECT
        order_id,
        customer_id,
        amount,
        updated_at,
        source_file,
        batch_id
    FROM bronze.sales_raw
    WHERE source_file = %s
    """,
    conn,
    params=("sales_2026_10_28.csv",)
)

print("Bronze records:", len(bronze_df))
display(bronze_df.head())

PostgreSQL password:  ········


PostgreSQL connected successfully!


C:\Users\Ajay Patil\AppData\Local\Temp\ipykernel_10408\3940756303.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  bronze_df = pd.read_sql_query(


Bronze records: 100000


,order_id,customer_id,amount,updated_at,source_file,batch_id
0,100001,C07271,7502.72,2026-10-07 04:14:00,sales_2026_10_28.csv,sales_2026_10_28
1,100002,C00861,1919.00,2026-10-06 03:07:00,sales_2026_10_28.csv,sales_2026_10_28
2,100003,C05391,22306.40,2026-10-11 01:31:00,sales_2026_10_28.csv,sales_2026_10_28
3,100004,C05192,11696.04,2026-10-08 22:45:00,sales_2026_10_28.csv,sales_2026_10_28
4,100005,C05735,46771.06,2026-10-04 17:54:00,sales_2026_10_28.csv,sales_2026_10_28


In [10]:
# Standardize data types
silver_df = bronze_df.copy()

silver_df["order_id"] = pd.to_numeric(
    silver_df["order_id"], errors="coerce"
).astype("Int64")

silver_df["customer_id"] = (
    silver_df["customer_id"]
    .astype("string")
    .str.strip()
    .str.upper()
)

silver_df["amount"] = pd.to_numeric(
    silver_df["amount"], errors="coerce"
)

silver_df["updated_at"] = pd.to_datetime(
    silver_df["updated_at"], errors="coerce"
)

# Separate invalid records
invalid_mask = (
    silver_df["order_id"].isna()
    | silver_df["customer_id"].isna()
    | silver_df["customer_id"].eq("")
    | silver_df["amount"].isna()
    | silver_df["updated_at"].isna()
    | (silver_df["amount"] < 0)
)

rejected_df = silver_df[invalid_mask].copy()
valid_df = silver_df[~invalid_mask].copy()

# Keep latest record for each order
valid_df = valid_df.sort_values(
    ["updated_at", "batch_id"],
    ascending=[False, True]
)

silver_df = valid_df.drop_duplicates(
    subset=["order_id"],
    keep="first"
).copy()

print("Valid unique records:", len(silver_df))
print("Rejected records:", len(rejected_df))
display(silver_df.head())

Valid unique records: 100000
Rejected records: 0


,order_id,customer_id,amount,updated_at,source_file,batch_id
179,100180,C00699,16539.71,2026-10-28 18:39:00,sales_2026_10_28.csv,sales_2026_10_28
22677,122678,C06881,44021.37,2026-10-28 18:39:00,sales_2026_10_28.csv,sales_2026_10_28
66405,166406,C08680,29188.77,2026-10-28 18:39:00,sales_2026_10_28.csv,sales_2026_10_28
40136,140137,C08978,17596.75,2026-10-28 18:38:00,sales_2026_10_28.csv,sales_2026_10_28
53191,153192,C08174,15807.09,2026-10-28 18:38:00,sales_2026_10_28.csv,sales_2026_10_28


In [11]:
upsert_sql = """
INSERT INTO silver.sales (
    order_id,
    customer_id,
    amount,
    updated_at,
    source_file,
    batch_id,
    processed_at
)
VALUES (%s, %s, %s, %s, %s, %s, CURRENT_TIMESTAMP)

ON CONFLICT (order_id)
DO UPDATE SET
    customer_id = EXCLUDED.customer_id,
    amount = EXCLUDED.amount,
    updated_at = EXCLUDED.updated_at,
    source_file = EXCLUDED.source_file,
    batch_id = EXCLUDED.batch_id,
    processed_at = CURRENT_TIMESTAMP

WHERE EXCLUDED.updated_at > silver.sales.updated_at;
"""

records = [
    (
        int(row.order_id),
        str(row.customer_id),
        float(row.amount),
        row.updated_at.to_pydatetime(),
        row.source_file,
        row.batch_id
    )
    for row in silver_df.itertuples(index=False)
]

try:
    with conn.cursor() as cur:
        cur.executemany(upsert_sql, records)
    conn.commit()
    print("Silver upsert completed:", len(records))

except Exception:
    conn.rollback()
    raise

Silver upsert completed: 100000


In [12]:
with conn.cursor() as cur:
    cur.execute("""
    select count(*) as total_records,count(distinct order_id) as uniueq_orders
    from silver.sales
    """)
    print(cur.fetchone())

(100000, 100000)
